# RAG experiment: chunking × retriever × return unit

Goal: pick the retrieval setup with evidence (quality vs cost), not by guess. See `NOTES.md` > "RAG experiment plan".

**Pipeline under test**
`pages -> chunk -> index (BM25 + dense) -> retrieve top-k -> expand (chunk / page / page ± j) -> LLM`

**Stage 1 (this notebook, no LLM):** does the right page come back? Cheap, repeatable.
**Stage 2 (bottom):** top configs + baselines end to end with cost logging.

| Factor | Levels |
|---|---|
| Chunk size (words) | page, 150, 300, 600 (overlap 20%) |
| Retriever | bm25, dense, hybrid (RRF) |
| top-k | 1, 3, 5 |
| Return unit | chunk, page, page ± 1 |
| Baselines | no retrieval, whole PDF in context |

Everything is scored at **page level** so differently chunked systems are comparable.

## 0. Config

In [ ]:
import json, re, math, itertools, hashlib, os, time
from pathlib import Path
import numpy as np, pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

ROOT = Path('.')                      # run from the UnravelCarbon folder
PAGES_JSON = ROOT/'build/ghg-protocol-revised.plain.json'   # from build step 1 (plain pdftotext)
EVAL_FILE  = ROOT/'build/eval_queries.jsonl'
CACHE = ROOT/'build/cache'; CACHE.mkdir(parents=True, exist_ok=True)

EMBED_MODEL = 'BAAI/bge-small-en-v1.5'   # FIXED across the experiment
SEED = 0

GRID = dict(
    chunk_words = ['page', 150, 300, 600],
    retriever   = ['bm25', 'dense', 'hybrid'],
    k           = [1, 3, 5],
    expand      = ['chunk', 'page', 'page+-1', 'page+-2'],
)
OVERLAP = 0.2
RRF_K = 60

## 1. Load pages

In [ ]:
pages = json.load(open(PAGES_JSON))
PAGE_TEXT = {p['page']: p['text'] for p in pages}   # 1-based PDF page index
print(len(PAGE_TEXT), 'pages;', sum(len(t.split()) for t in PAGE_TEXT.values()), 'words')

## 2. Chunkers
Each chunk keeps the **set of PDF pages** its words came from, so any chunk maps back to pages.
`'page'` = one chunk per page (the current plan). Word counts approximate tokens (~1.3 tokens/word).

In [ ]:
def chunk_pages(chunk_words):
    chunks = []
    if chunk_words == 'page':
        for pg, t in PAGE_TEXT.items():
            chunks.append(dict(id=len(chunks), pages=[pg], text=t))
        return chunks
    words, wpage = [], []
    for pg, t in PAGE_TEXT.items():
        w = t.split(); words += w; wpage += [pg]*len(w)
    step = max(1, int(chunk_words*(1-OVERLAP)))
    for s in range(0, len(words), step):
        e = min(len(words), s+chunk_words)
        chunks.append(dict(id=len(chunks), pages=sorted(set(wpage[s:e])), text=' '.join(words[s:e])))
        if e == len(words): break
    return chunks

for cw in GRID['chunk_words']:
    ch = chunk_pages(cw)
    print(cw, len(ch), 'chunks; avg words', int(np.mean([len(c['text'].split()) for c in ch])))

## 3. Retrievers
All return a **ranked list of chunk ids with scores**. Dense embeddings are cached on disk (one-time cost, and logged).

In [ ]:
tok = lambda s: re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", s.lower())
_model = None
def model():
    global _model
    if _model is None: _model = SentenceTransformer(EMBED_MODEL)
    return _model

class Index:
    def __init__(self, chunk_words):
        self.cw = chunk_words
        self.chunks = chunk_pages(chunk_words)
        self.bm25 = BM25Okapi([tok(c['text']) for c in self.chunks])
        key = hashlib.md5((EMBED_MODEL+str(chunk_words)+str(len(self.chunks))).encode()).hexdigest()[:10]
        f = CACHE/f'emb_{key}.npy'
        t0 = time.time()
        if f.exists(): self.emb = np.load(f); self.embed_secs = 0.0
        else:
            self.emb = model().encode([c['text'] for c in self.chunks], normalize_embeddings=True, batch_size=32, show_progress_bar=False)
            np.save(f, self.emb); self.embed_secs = time.time()-t0

    def scores(self, q, retriever):
        if retriever == 'bm25':
            return np.array(self.bm25.get_scores(tok(q)))
        qe = model().encode([q], normalize_embeddings=True)[0]
        dense = self.emb @ qe
        if retriever == 'dense': return dense
        # hybrid: reciprocal rank fusion of the two rankings
        b = np.array(self.bm25.get_scores(tok(q)))
        rb = np.empty(len(b)); rb[np.argsort(-b)] = np.arange(len(b))
        rd = np.empty(len(b)); rd[np.argsort(-dense)] = np.arange(len(b))
        return 1/(RRF_K+rb+1) + 1/(RRF_K+rd+1)

    def dense_top(self, q):
        return float((self.emb @ model().encode([q], normalize_embeddings=True)[0]).max())

INDEX = {}
def get_index(cw):
    if cw not in INDEX: INDEX[cw] = Index(cw)
    return INDEX[cw]

## 4. Retrieve and expand
`expand`:
- `chunk`: only the pages the chunk touches
- `page`: the full page(s) of each hit (what the LLM would read)
- `page+-1`: neighbours ±j (j=1) around each hit page, **merged and de-duplicated**

Also returns a ranked **page list** (best chunk first) for MRR, and a token-cost proxy for the context returned.

In [ ]:
def retrieve(q, cw, retriever, k, expand):
    j = int(expand.split('-')[1]) if expand.startswith('page+-') else 0
    ix = get_index(cw); s = ix.scores(q, retriever)
    top = np.argsort(-s)[:k]
    ranked_pages = []
    for ci in top:
        for p in ix.chunks[ci]['pages']:
            if p not in ranked_pages: ranked_pages.append(p)
    if expand in ('chunk','page'): pages_ret = list(ranked_pages)
    elif j:
        pages_ret = sorted({p+d for p in ranked_pages for d in range(-j, j+1) if 1 <= p+d <= len(PAGE_TEXT)})
    if expand == 'chunk': ctx_words = sum(len(ix.chunks[ci]['text'].split()) for ci in top)
    else: ctx_words = sum(len(PAGE_TEXT[p].split()) for p in pages_ret)
    return dict(ranked_pages=ranked_pages, pages=pages_ret, ctx_tokens=int(ctx_words*1.3),
                top_score=float(s[top[0]]), dense_max=ix.dense_top(q))

## 5. Evaluation set
JSONL, one query per line: `query`, `type` (rule | paraphrase | keyword | peer | negative), `gold_pages` (list; empty for negative), `key_points`, `split` (dev | test).
The file is produced by `build/make_eval.py`, which checks every gold page against an exact quote from that page and every negative against the full text. **Do not edit the JSONL by hand; edit make_eval.py and re-run it.** Tune only on `dev`; report on `test`. `also_relevant` pages are partial matches that are not penalised in precision.

In [ ]:
SEED_ROWS = [
 dict(query='How are Scope 2 emissions defined?', type='rule', gold_pages=[29,27], key_points=['generation of purchased electricity','consumed in owned or controlled equipment'], split='dev'),
 dict(query='Which energy types count as indirect Scope 2 sources?', type='rule', gold_pages=[43], key_points=['electricity','heat','steam'], split='dev'),
 dict(query='How should Scope 2 emissions be calculated and which emission factors are acceptable?', type='rule', gold_pages=[44], key_points=['supplier-specific, local grid, or other published factors'], split='dev'),
 dict(query='What information must a public GHG report include?', type='rule', gold_pages=[65], key_points=['reporting period','scopes','six gases','methodologies','exclusions'], split='test'),
 dict(query='How are renewable energy certificates treated?', type='keyword', gold_pages=[63,61], key_points=['RECs reported separately'], split='dev'),
 dict(query='electricity bought from a utility', type='paraphrase', gold_pages=[29,43], key_points=['purchased electricity'], split='test'),
 dict(query='Which scope covers employee business travel?', type='rule', gold_pages=[], key_points=['TODO: locate pages (28,31,33,45 are candidates)'], split='dev'),
 dict(query='What is the market-based method for Scope 2?', type='negative', gold_pages=[], key_points=['not in 2004 PDF'], split='dev'),
 dict(query='location-based vs market-based dual reporting', type='negative', gold_pages=[], key_points=['not in 2004 PDF'], split='test'),
]
if not EVAL_FILE.exists():
    EVAL_FILE.write_text('\n'.join(json.dumps(r) for r in SEED_ROWS)+'\n')
EVAL = [json.loads(l) for l in EVAL_FILE.read_text().splitlines() if l.strip()]
EVAL_POS = [e for e in EVAL if e['type']!='negative' and e['gold_pages']]
EVAL_NEG = [e for e in EVAL if e['type']=='negative']
print(len(EVAL), 'queries;', len(EVAL_POS), 'scorable positives;', len(EVAL_NEG), 'negatives')

## 6. Metrics (page level)
- **Recall@k**: share of gold pages present in the returned pages (any gold page counts as a hit for `hit@k`)
- **MRR**: reciprocal rank of the first gold page in the ranked page list (before expansion)
- **Precision**: returned pages that are gold or marked `also_relevant`
- **ctx_tokens**: context size handed to the LLM (cost proxy)
- **No-match accuracy**: negatives where the best dense cosine is below threshold `tau` (tuned on dev negatives vs dev positives)

In [ ]:
def score_query(e, r):
    gold = set(e['gold_pages']); rel = gold | set(e.get('also_relevant', [])); got = set(r['pages'])
    rr = next((1/(i+1) for i,p in enumerate(r['ranked_pages']) if p in gold), 0.0)
    return dict(recall=len(gold&got)/len(gold), hit=float(bool(gold&got)), mrr=rr,
                precision=len(rel&got)/max(1,len(got)), ctx_tokens=r['ctx_tokens'])

def run_config(cw, retriever, k, expand, split=None):
    rows = []
    for e in EVAL_POS:
        if split and e['split']!=split: continue
        rows.append(score_query(e, retrieve(e['query'], cw, retriever, k, expand)))
    return pd.DataFrame(rows).mean().to_dict() if rows else {}

## 7. Run the grid (stage 1)

In [ ]:
results = []
for cw, ret, k, ex in itertools.product(*GRID.values()):
    # 'chunk' vs 'page' return are identical when the chunk IS a page
    if cw=='page' and ex=='chunk': continue
    res = run_config(cw, ret, k, ex)
    results.append(dict(chunk=cw, retriever=ret, k=k, expand=ex, **res))
R = pd.DataFrame(results).sort_values(['recall','mrr'], ascending=False)
R.to_csv(ROOT/'build/stage1_results.csv', index=False)
R.head(15)

### Reading the table
Look at the **recall vs ctx_tokens** trade-off, not recall alone. A config that reaches the same recall with half the tokens wins. With ~116 pages and ~30–50 queries, small differences are noise; report that honestly.

In [ ]:
best = R.groupby('retriever')[['recall','mrr','ctx_tokens']].max()
display(best)
display(R.pivot_table(index='chunk', columns='retriever', values='recall', aggfunc='max'))
display(R.pivot_table(index='expand', columns='k', values='recall', aggfunc='mean'))

## 8. No-match (negative queries)

In [ ]:
pos_scores = [get_index('page').dense_top(e['query']) for e in EVAL_POS if e['split']=='dev']
neg_scores = [get_index('page').dense_top(e['query']) for e in EVAL_NEG if e['split']=='dev']
print('dev positives max-cosine:', np.round(pos_scores,3))
print('dev negatives max-cosine:', np.round(neg_scores,3))
# choose tau between the two groups (needs enough queries to be meaningful)
tau = (min(pos_scores)+max(neg_scores))/2 if pos_scores and neg_scores else 0.5
test_neg = [get_index('page').dense_top(e['query']) for e in EVAL_NEG if e['split']=='test']
print('tau =', round(tau,3), '| test no-match accuracy:', np.mean([s<tau for s in test_neg]) if test_neg else 'n/a')

## 9. Baselines
- **No retrieval**: LLM answers from memory (expect confident, uncited answers).
- **Whole PDF in context**: ~60–80k tokens per query; compute exact count below.

In [ ]:
whole_tokens = int(sum(len(t.split()) for t in PAGE_TEXT.values())*1.3)
print('whole-document context ≈', whole_tokens, 'tokens/query')

## 10. Stage 2: end to end (top configs + baselines)
Fix LLM, temperature 0, top-k. Repeat each generation ~3 times and report the spread. Log **every run** with the system-prompt version.
Fill `call_llm` with the chosen provider; keep the logging schema.

In [ ]:
SYSTEM_PROMPT_VERSION = 'v0'
RUN_LOG = []

def call_llm(system, user):
    raise NotImplementedError('plug in the chosen model; return (text, input_tokens, output_tokens)')

def answer(q, method, cfg=None):
    t0 = time.time()
    if method == 'no_retrieval': ctx = ''
    elif method == 'whole_pdf':  ctx = '\n\n'.join(f'[p.{p}]\n{t}' for p,t in PAGE_TEXT.items())
    else:
        r = retrieve(q, **cfg); ctx = '\n\n'.join(f'[p.{p}]\n{PAGE_TEXT[p]}' for p in r['pages'])
    text, tin, tout = call_llm('<system prompt here>', f'Context:\n{ctx}\n\nQuestion: {q}')
    RUN_LOG.append(dict(prompt_version=SYSTEM_PROMPT_VERSION, method=method, cfg=cfg, query=q,
                        in_tokens=tin, out_tokens=tout, latency=time.time()-t0, answer=text))
    return text

### Stage 2 scoring (manual + LLM judge)
Per run: key-point coverage, **citation accuracy** (does the cited page contain the claim?), invented content, then dollars from tokens.
Final table: **method × question → cost, quality**.

In [ ]:
# cost table template (fill prices for the chosen model)
PRICE_IN, PRICE_OUT = 0.0, 0.0   # $ per 1M tokens
def run_cost(r): return (r['in_tokens']*PRICE_IN + r['out_tokens']*PRICE_OUT)/1e6
# pd.DataFrame(RUN_LOG).assign(cost=lambda d: d.apply(run_cost, axis=1)).groupby('method')[['in_tokens','cost','latency']].mean()

## Protocol checklist
- [ ] Extraction fixed first (plain `pdftotext`; re-check pages with tables/figures)
- [ ] Eval set 30–50 queries incl. paraphrases, keyword, peer, negatives; business-travel gold pages located
- [ ] One factor at a time; same embedding model, seed, k grid
- [ ] Tune on `dev`, report on `test`
- [ ] Report indexing time (`embed_secs`) and tokens alongside quality
- [ ] Limitations: tiny corpus (noise), tables in peer PDFs, two-column interleaving